In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SOURCE = Path(os.environ.get("RESEARCH_REPORT_SOURCE", ROOT / "data" / "research_report" / "forecast_stk_20240101_20241231.jsonl"))
TRADING_CALENDAR = Path(os.environ.get("RESEARCH_REPORT_TRADING_CALENDAR", ROOT / "data" / "RTN_daily" / "rtn_1d.parquet"))
from src.trading_calendar import align_to_trading_day, load_trading_dates, parse_market_timestamps
trading_dates = load_trading_dates(TRADING_CALENDAR)
df = pd.read_json(SOURCE, lines=True)
df["STOCK_CODE"] = df["STOCK_CODE"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(6)
df["PUBLISH_TIMESTAMP"] = parse_market_timestamps(df["CREATE_DATE"])
df["PUBLISH_DATE"] = df["PUBLISH_TIMESTAMP"].dt.normalize()
df["AVAILABLE_DATE"] = align_to_trading_day(df["PUBLISH_TIMESTAMP"], trading_dates)

In [ ]:
df.head()

In [ ]:
print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())

# 数据质量彻查（从第 4 个 cell 开始）

目标：检查 `RESEARCH_REPORT_SOURCE`（默认仓库 `data/research_report/`）中的研报数据质量。

> 前 3 个 cell 保持不变：`read_json` 加载 `df` → `df.head()` → 打印 `shape` 与 `columns`。
> 本 notebook 需要按顺序从上到下运行，后面的 cell 都依赖 `df` 与前面 cell 中定义的字段分组常量。


In [ ]:
# ============================================================
# 0. 全局配置：导入、字段分组、字段含义
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# 使用系统中文字体，避免图上中文显示为方框
plt.rcParams["font.sans-serif"] = ["WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("DataFrame shape:", df.shape)
print("内存占用(deep): {:.1f} MB".format(df.memory_usage(deep=True).sum() / 1024**2))

# ---- 后续分析实际使用的字段分组 ----
forecast_cols = ["FORECAST_OR", "FORECAST_OP", "FORECAST_TP", "FORECAST_NP",
                 "FORECAST_EPS", "FORECAST_DPS", "FORECAST_RD", "FORECAST_PE",
                 "FORECAST_ROE", "FORECAST_EV_EBITDA"]
price_cols    = ["TARGET_PRICE_CEILING", "TARGET_PRICE_FLOOR", "CURRENT_PRICE"]
cat_cols      = ["REPORT_TYPE", "ATTENTION", "RELIABILITY", "LANGUAGE", "CURRENCY"]

field_meaning = {
    "ID": "记录唯一ID",
    "STOCK_CODE": "股票代码",
    "STOCK_NAME": "股票名称",
    "TITLE": "报告标题",
    "CONTENT": "报告正文内容",
    "REPORT_TYPE": "报告类型(编码)",
    "RELIABILITY": "可靠度评分",
    "ORGAN_NAME": "发布机构",
    "AUTHOR_NAME": "分析师姓名(逗号分隔)",
    "CREATE_DATE": "报告创建日期",
    "REPORT_YEAR": "报告对应年度",
    "REPORT_QUARTER": "报告对应季度",
    "FORECAST_OR": "预测营业收入",
    "FORECAST_OP": "预测营业利润",
    "FORECAST_TP": "预测利润总额",
    "FORECAST_NP": "预测净利润",
    "FORECAST_EPS": "预测每股收益 EPS",
    "FORECAST_DPS": "预测每股股利 DPS",
    "FORECAST_RD": "预测研发费用",
    "FORECAST_PE": "预测市盈率 PE",
    "FORECAST_ROE": "预测净资产收益率 ROE",
    "FORECAST_EV_EBITDA": "预测 EV/EBITDA",
    "ORGAN_RATING_CODE": "机构评级编码",
    "ORGAN_RATING_CONTENT": "机构评级文案",
    "GG_RATING_CODE": "个股评级编码",
    "GG_RATING_CONTENT": "个股评级文案",
    "TARGET_PRICE_CEILING": "目标价上限",
    "TARGET_PRICE_FLOOR": "目标价下限",
    "CURRENT_PRICE": "当前股价",
    "CURRENCY": "币种",
    "LANGUAGE": "语言(编码)",
    "ATTENTION": "报告关注标签",
}
pd.DataFrame({"字段含义": field_meaning}).reindex(df.columns)


## 1. 字段类型 / 缺失 / 唯一值总览


In [ ]:
# ============================================================
# 1. 字段类型 / 缺失 / 唯一值总览
# ============================================================
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
})
overview.sort_values(["null_pct", "n_unique"], ascending=[False, False])


## 2. 主键与唯一性 / 重复行


In [ ]:
# ============================================================
# 2. 主键与唯一性 / 重复行
# ============================================================
print("总行数:", len(df))
print("ID 唯一性 is_unique =", df["ID"].is_unique, "| ID 唯一值数 =", df["ID"].nunique())
print("完全重复的行数 =", int(df.duplicated().sum()))
print("按 ID 分组后行数>1 的 ID 个数 =", int((df.groupby("ID").size() > 1).sum()))
print("去掉 ID 列后仍完全重复的记录数 =", int(df.drop(columns=["ID"]).duplicated().sum()))

# 股票代码 vs 股票名称 的映射关系
code2name = df.groupby("STOCK_CODE")["STOCK_NAME"].nunique()
name2code = df.groupby("STOCK_NAME")["STOCK_CODE"].nunique()
print("STOCK_CODE 唯一值 = %d，其中 1 个代码对应多个名称 的个数 = %d" % (
    df["STOCK_CODE"].nunique(), int((code2name > 1).sum())))
print("STOCK_NAME 唯一值 = %d，其中 1 个名称对应多个代码 的个数 = %d" % (
    df["STOCK_NAME"].nunique(), int((name2code > 1).sum())))

if (code2name > 1).any():
    print("一个代码对应多个名称的示例：")
    for code in code2name[code2name > 1].index[:10]:
        names = sorted(set(df.loc[df["STOCK_CODE"] == code, "STOCK_NAME"]))
        print("  代码", code, "->", names)
if (name2code > 1).any():
    print("一个名称对应多个代码的示例：")
    for name in name2code[name2code > 1].index[:10]:
        codes = sorted(set(df.loc[df["STOCK_NAME"] == name, "STOCK_CODE"]))
        print("  名称", name, "->", codes)


## 3. 时间维度（CREATE_DATE）


In [ ]:
# ============================================================
# 3. 时间维度 CREATE_DATE
# ============================================================
dt = df["PUBLISH_TIMESTAMP"]
print("能否全部解析为日期:", bool(dt.notna().all()), "| 解析失败行数:", int(dt.isna().sum()))
print("时间范围:", dt.min(), "→", dt.max())
print("跨度天数:", (dt.max() - dt.min()).days)
print("不同日期数:", dt.nunique())

monthly = dt.dt.to_period("M").value_counts().sort_index()
print("\n按月报告数：")
print(monthly)

plt.figure(figsize=(14, 4))
monthly.plot(kind="bar", color="steelblue")
plt.title("报告数量按月分布")
plt.ylabel("报告数")
plt.xlabel("月份")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 4. 文本字段（TITLE / CONTENT）


In [ ]:
# ============================================================
# 4.1 文本字段长度分布
# ============================================================
title_len = df["TITLE"].str.len()
print("TITLE：无缺失 =", bool(df["TITLE"].notna().all()),
      "| 长度 min/中位/均值/max = %d / %.1f / %.1f / %d" % (
          title_len.min(), title_len.median(), title_len.mean(), title_len.max()))

content_non_null = df["CONTENT"].dropna()
print("\nCONTENT：非空 %d / %d 条，缺失 %d 条 (%.2f%%)" % (
    content_non_null.shape[0], df.shape[0],
    int(df["CONTENT"].isna().sum()), df["CONTENT"].isna().mean() * 100))

clen = content_non_null.str.len()
print("CONTENT(非空) 长度：min/中位/均值/max = %d / %.1f / %.1f / %d" % (
    clen.min(), clen.median(), clen.mean(), clen.max()))
print("CONTENT 长度分位数：")
print(clen.quantile([0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(title_len, bins=50, color="steelblue")
axes[0].set_title("TITLE 长度分布")
axes[0].set_xlabel("字符数")
axes[1].hist(clen, bins=100, color="coral")
axes[1].set_title("CONTENT 长度分布")
axes[1].set_xlabel("字符数")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4.2 文本重复情况 与 样本查看
# ============================================================
print("TITLE  唯一值 %d / 总行数 %d，重复标题行数 = %d" % (
    df["TITLE"].nunique(), len(df), int(df["TITLE"].duplicated().sum())))
print("CONTENT 唯一值(非空) %d / 非空条数 %d，重复正文条数 = %d" % (
    content_non_null.nunique(), len(content_non_null),
    int(content_non_null.duplicated().sum())))

# 把标题里的“股票名：”前缀去掉，看最常出现的报告主体
title_core = df["TITLE"].str.replace(r"^[^：:]{1,20}[：:]", "", regex=True)
print("\n最常见的标题主体 Top 15：")
print(title_core.value_counts().head(15).to_string())

longest_idx = clen.idxmax()
shortest_idx = clen.idxmin()
print("\n--- 最长正文样本 (index=%d, %d 字符) ---" % (longest_idx, clen[longest_idx]))
print("标题:", df.loc[longest_idx, "TITLE"])
print("正文前 1500 字:")
print(df.loc[longest_idx, "CONTENT"][:1500])
print("\n--- 最短正文样本 (index=%d, %d 字符) ---" % (shortest_idx, clen[shortest_idx]))
print("标题:", df.loc[shortest_idx, "TITLE"])
print("正文(repr):", repr(df.loc[shortest_idx, "CONTENT"]))


## 5. 分类字段取值分布


In [ ]:
# ============================================================
# 5. 分类字段取值分布
# ============================================================
for c in cat_cols:
    print("=" * 70)
    print("%s  dtype=%s  唯一值数=%d" % (c, df[c].dtype, df[c].nunique(dropna=True)))
    print(df[c].value_counts(dropna=False).to_string())
    print()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.ravel()
for i, c in enumerate(cat_cols):
    df[c].value_counts(dropna=False).plot(kind="bar", ax=axes[i], color="steelblue")
    axes[i].set_title(c)
    axes[i].tick_params(axis="x", rotation=45)
axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 6. 评级字段：编码 ↔ 文案对应关系


In [ ]:
# ============================================================
# 6. 评级字段：编码 与 文案 的对应关系
# ============================================================
for cc, content_col in [("ORGAN_RATING_CODE", "ORGAN_RATING_CONTENT"),
                        ("GG_RATING_CODE", "GG_RATING_CONTENT")]:
    print("=" * 70)
    print(cc, "<->", content_col)
    mapping = (df[[cc, content_col]].groupby(cc)[content_col]
               .agg(lambda s: sorted(set(s))).sort_index())
    for codev, contents in mapping.items():
        print("  %-6s -> %s" % (codev, contents))
    print()

print("=" * 70)
print("评级文案取值分布：")
for content_col in ["ORGAN_RATING_CONTENT", "GG_RATING_CONTENT"]:
    print("\n", content_col)
    print(df[content_col].value_counts(dropna=False).to_string())


## 7. 机构 / 分析师 / 股票分布


In [ ]:
# ============================================================
# 7. 机构 / 分析师 / 股票分布
# ============================================================
print("机构数 =", df["ORGAN_NAME"].nunique())
print("Top20 机构：")
print(df["ORGAN_NAME"].value_counts().head(20).to_string())

print("\n股票代码数 =", df["STOCK_CODE"].nunique(), "| 股票名称数 =", df["STOCK_NAME"].nunique())
print("Top20 股票(名称) 报告数：")
print(df["STOCK_NAME"].value_counts().head(20).to_string())

# 分析师拆分统计（AUTHOR_NAME 以逗号分隔）
authors = df["AUTHOR_NAME"].dropna().str.split(",", expand=False).explode().str.strip()
print("\n分析师去重人数 =", authors.nunique())
print("Top20 分析师：")
print(authors.value_counts().head(20).to_string())


## 8. 数值字段（预测值 & 价格）分布


In [ ]:
# ============================================================
# 8.1 数值字段统计描述
# ============================================================
num_cols = forecast_cols + price_cols
desc = df[num_cols].describe().T
desc["missing_rate"] = df[num_cols].isna().mean().round(4)
desc


In [ ]:
# ============================================================
# 8.2 数值字段分布直方图（裁剪到 1%~99% 便于观察）
# ============================================================
ncols = len(num_cols)
nrow = int(np.ceil(ncols / 4))
fig, axes = plt.subplots(nrow, 4, figsize=(16, 2.6 * nrow))
axes = np.array(axes).ravel()
for i, c in enumerate(num_cols):
    s = df[c].dropna()
    if len(s) == 0:
        axes[i].set_title("%s (全空)" % c)
        axes[i].axis("off")
        continue
    lo, hi = s.quantile(0.01), s.quantile(0.99)
    if lo == hi:
        lo, hi = s.min(), s.max()
    axes[i].hist(s, bins=60, color="steelblue")
    axes[i].set_xlim(lo, hi)
    axes[i].set_title("%s (n=%d)" % (c, len(s)))
    axes[i].tick_params(axis="x", rotation=30)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 8.3 数值字段相关性
# ============================================================
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.title("数值字段 Pearson 相关系数")
plt.tight_layout()
plt.show()
corr.round(2)


## 9. 字段交叉关系


In [ ]:
# ============================================================
# 9.1 报告类型 × 关注标签 × 是否有正文
# ============================================================
print("报告类型(REPORT_TYPE) 取值分布：")
print(df["REPORT_TYPE"].value_counts(dropna=False).to_string())
print("\n关注标签(ATTENTION) 取值分布：")
print(df["ATTENTION"].value_counts(dropna=False).to_string())
print("\n报告类型 × 关注标签 交叉表：")
print(pd.crosstab(df["REPORT_TYPE"], df["ATTENTION"], margins=True))

tmp = df.copy()
tmp["HAS_CONTENT"] = tmp["CONTENT"].notna()
print("\n报告类型 × 是否有正文 交叉表：")
print(pd.crosstab(tmp["REPORT_TYPE"], tmp["HAS_CONTENT"], margins=True))


In [ ]:
# ============================================================
# 9.2 财报年度 × 季度 分布
# ============================================================
print("REPORT_YEAR × REPORT_QUARTER 交叉表（报告对应的财报期）：")
print(pd.crosstab(df["REPORT_YEAR"], df["REPORT_QUARTER"], margins=True))

fig, ax = plt.subplots(figsize=(12, 4))
ct = pd.crosstab(df["REPORT_YEAR"], df["REPORT_QUARTER"])
ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab10")
ax.set_title("各财报年度 × 季度 报告数量")
ax.set_ylabel("报告数")
ax.set_xlabel("REPORT_YEAR")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. 原始数据抽样查看


In [ ]:
# ============================================================
# 10. 原始数据抽样查看（逐字段打印若干条完整记录）
# ============================================================
sample = df.sample(n=3, random_state=42)

for i, (idx, row) in enumerate(sample.iterrows(), 1):
    print("=" * 80)
    print("样本 %d / 3   (原始 index=%d, ID=%s)" % (i, idx, row["ID"]))
    print("-" * 80)
    for col in df.columns:
        val = row[col]
        if col == "CONTENT" and isinstance(val, str) and len(val) > 800:
            val = val[:800] + "  ......[截断]"
        print("%-24s : %s" % (col, val))
    print()


## 11. 结论摘要（自动汇总）


In [ ]:
# ============================================================
# 11. 结论摘要（自动汇总）
# ============================================================
dt_all = df["PUBLISH_TIMESTAMP"]
authors_all = df["AUTHOR_NAME"].dropna().str.split(",", expand=False).explode().str.strip()

lines = []
lines.append("文件: %s" % SOURCE)
lines.append("总记录数: %d 行 × %d 列" % df.shape)
lines.append("时间范围: %s ~ %s (跨度 %d 天)" % (
    dt_all.min(), dt_all.max(), (dt_all.max() - dt_all.min()).days))
lines.append("股票: 代码 %d 个 / 名称 %d 个；机构 %d 家；分析师 %d 位" % (
    df["STOCK_CODE"].nunique(), df["STOCK_NAME"].nunique(),
    df["ORGAN_NAME"].nunique(), authors_all.nunique()))
lines.append("ID 唯一: %s；完全重复行: %d；去掉 ID 后重复: %d" % (
    df["ID"].is_unique, int(df.duplicated().sum()),
    int(df.drop(columns=["ID"]).duplicated().sum())))
lines.append("正文(CONTENT)缺失率: %.2f%%" % (df["CONTENT"].isna().mean() * 100))
lines.append("目标价上限缺失率: %.2f%%；币种(CURRENCY)缺失率: %.2f%%" % (
    df["TARGET_PRICE_CEILING"].isna().mean() * 100,
    df["CURRENCY"].isna().mean() * 100))
lines.append("财报期: 年度取值 %s；季度取值 %s" % (
    sorted(df["REPORT_YEAR"].dropna().unique().tolist()),
    sorted(df["REPORT_QUARTER"].dropna().unique().tolist())))
lines.append("报告类型取值: %s" % sorted(df["REPORT_TYPE"].dropna().unique().tolist()))
lines.append("关注标签取值: %s" % sorted(df["ATTENTION"].dropna().unique().tolist()))
lines.append("评级文案取值: %s" % sorted(df["ORGAN_RATING_CONTENT"].dropna().unique().tolist()))
print("\n".join(lines))
